# Магические руны Элдории
**Задача:** бинарная классификация 5-символьных строк (F, W, E) → spell (0/1)

**Подход:**
1. EDA — анализ пространства, распределения, позиционных паттернов
2. Feature engineering — one-hot позиций, счётчики символов, числовые значения
3. Модели — Decision Tree, GradientBoosting, перекрёстная валидация
4. Генерация answers.csv

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

train = pd.read_csv("data/train_runes.csv")
test = pd.read_csv("data/test_runes.csv")
example = pd.read_csv("data/example.csv")

print(f"Train: {len(train)} rows | Test: {len(test)} rows")
print(f"Total unique runes: {len(set(train['rune']) | set(test['rune']))} / 3^5 = 243")
print(f"\nTrain spell distribution:\n{train['spell'].value_counts().to_string()}")

## EDA: позиционные паттерны и счётчики символов

In [ ]:
# Доля spell=1 по символу на каждой позиции
print("=== Доля spell=1 по (позиция, символ) ===\n")
for i in range(5):
    train[f'pos{i}'] = train['rune'].str[i]
    ct = pd.crosstab(train[f'pos{i}'], train['spell'], margins=True)
    ct['spell=1 ratio'] = ct[1] / (ct[0] + ct[1])
    print(f"Position {i}:")
    print(ct[['spell=1 ratio']].to_string(), "\n")

# Счётчики символов
print("=== Доля spell=1 по количеству символа ===\n")
for ch in 'FWE':
    train[f'cnt_{ch}'] = train['rune'].apply(lambda x: x.count(ch))
    ct = pd.crosstab(train[f'cnt_{ch}'], train['spell'])
    ct['spell=1 ratio'] = ct[1] / (ct[0] + ct[1])
    print(f"Count of {ch}:")
    print(ct[['spell=1 ratio']].to_string(), "\n")

## Feature Engineering

Признаки:
- **One-hot:** 15 бинарных признаков `pos_i == ch` для каждой позиции и символа
- **Counts:** количество каждого символа в строке
- **Numeric:** числовое значение символа на каждой позиции (F=0, W=1, E=2)

In [ ]:
ENC = {'F': 0, 'W': 1, 'E': 2}

def make_features(df):
    """Создаёт матрицу признаков из колонки rune."""
    feats = pd.DataFrame(index=df.index)

    # One-hot: pos_i == char
    for i in range(5):
        for ch in 'FWE':
            feats[f'p{i}_{ch}'] = (df['rune'].str[i] == ch).astype(int)

    # Counts
    for ch in 'FWE':
        feats[f'cnt_{ch}'] = df['rune'].apply(lambda x: x.count(ch))

    # Numeric per position
    for i in range(5):
        feats[f'val_{i}'] = df['rune'].str[i].map(ENC)

    return feats

feat_names = None

X_train = make_features(train)
y_train = train['spell'].values
X_test = make_features(test)

feat_names = X_train.columns.tolist()
print(f"Features: {len(feat_names)}")
X_train.head()

## Модели и кросс-валидация

In [ ]:
# Decision Tree — подбор глубины
print("=== Decision Tree: подбор max_depth ===\n")
results = []
for depth in [3, 4, 5, 6, 7, 8, 10, None]:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    cv = cross_val_score(clf, X_train, y_train, cv=10, scoring='accuracy')
    clf.fit(X_train, y_train)
    train_acc = clf.score(X_train, y_train)
    results.append((depth, cv.mean(), cv.std(), train_acc))
    print(f"depth={str(depth):>4s}:  CV={cv.mean():.4f} ± {cv.std():.4f}  |  train={train_acc:.4f}")

print("\n=== GradientBoosting ===\n")
gbt = GradientBoostingClassifier(n_estimators=100, random_state=42)
cv_gbt = cross_val_score(gbt, X_train, y_train, cv=10, scoring='accuracy')
gbt.fit(X_train, y_train)
print(f"GBT:   CV={cv_gbt.mean():.4f} ± {cv_gbt.std():.4f}  |  train={gbt.score(X_train, y_train):.4f}")

## Интерпретация: дерево решений (depth=6)

In [ ]:
# Визуализация лучшего дерева
tree_clf = DecisionTreeClassifier(max_depth=6, random_state=42)
tree_clf.fit(X_train, y_train)
print(export_text(tree_clf, feature_names=feat_names))

## Проверка согласованности моделей и генерация ответа

In [ ]:
# Предсказания обеих моделей
preds_tree = tree_clf.predict(X_test)
preds_gbt = gbt.predict(X_test)

agree = (preds_tree == preds_gbt).sum()
print(f"Совпадение Tree vs GBT: {agree}/{len(preds_gbt)}")
print(f"  Tree: spell=1 → {preds_tree.sum()}, spell=0 → {len(preds_tree) - preds_tree.sum()}")
print(f"  GBT:  spell=1 → {preds_gbt.sum()}, spell=0 → {len(preds_gbt) - preds_gbt.sum()}")

In [ ]:
# Сохранение ответа (используем GBT — CV=1.0)
answers = pd.DataFrame({
    'rune': test['rune'],
    'spell': preds_gbt.astype(int)
})
answers.to_csv("data/answers.csv", index=False)

print(f"Сохранено {len(answers)} предсказаний в data/answers.csv")
print(f"\nПервые 5 строк:")
answers.head()